# VocaLip Thai — Notebook 04: Temporal Sensitivity Diagnostic
ไม่ retrain โมเดล ทดสอบ Validation 354 clips ด้วย Original / Reverse / Shuffle / Static และวัด Accuracy, Macro-F1, Top-3, Top-1 agreement, L1 และ Jensen-Shannon divergence.

In [ ]:
# CELL 1 — Setup
import json,time
from pathlib import Path
import numpy as np,pandas as pd,torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from tqdm import tqdm
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT=Path(r"D:\Face_detection\dataset\word_level_v5\video001")
MANIFEST=ROOT/"word_train_manifest_v5_baseline_clean.csv"
DEPLOY=ROOT/"models_03_ssl_finetune"/"03_vocalip_44words_deployment.pt"
OUT=ROOT/"models_04_temporal_diagnostic";OUT.mkdir(parents=True,exist_ok=True)
BATCH=8;AMP=DEVICE.type=="cuda";SEED=42
assert MANIFEST.exists(),MANIFEST
assert DEPLOY.exists(),DEPLOY
print("Device:",DEVICE);print("Model:",DEPLOY)

In [ ]:
# CELL 2 — Checkpoint
ckpt=torch.load(DEPLOY,map_location="cpu",weights_only=False)
print("Keys:",list(ckpt.keys()))
print("Config:",ckpt.get("model_config"))
print("Preprocessing:",ckpt.get("preprocessing"))
print("Recorded test:",ckpt.get("test_metrics"))
id2word={int(k):str(v) for k,v in ckpt["id_to_word"].items()}
assert len(id2word)==44

In [ ]:
# CELL 3 — Exact Notebook-03 architecture
class FrameCNN(nn.Module):
 def __init__(self):
  super().__init__();self.features=nn.Sequential(
   nn.Conv2d(3,32,3,padding=1,bias=False),nn.BatchNorm2d(32),nn.ReLU(True),nn.MaxPool2d(2),
   nn.Conv2d(32,64,3,padding=1,bias=False),nn.BatchNorm2d(64),nn.ReLU(True),nn.MaxPool2d(2),
   nn.Conv2d(64,96,3,padding=1,bias=False),nn.BatchNorm2d(96),nn.ReLU(True),nn.MaxPool2d(2),
   nn.Conv2d(96,128,3,padding=1,bias=False),nn.BatchNorm2d(128),nn.ReLU(True),nn.AdaptiveAvgPool2d((1,1)))
  self.projection=nn.Sequential(nn.Flatten(),nn.Linear(128,128),nn.ReLU(True),nn.Dropout(.10))
 def forward(self,x):return self.projection(self.features(x))
class Attn(nn.Module):
 def __init__(self):
  super().__init__();self.attention=nn.Sequential(nn.Linear(512,128),nn.Tanh(),nn.Linear(128,1))
 def forward(self,x):
  w=torch.softmax(self.attention(x).squeeze(-1),1);return (x*w.unsqueeze(-1)).sum(1),w
class Encoder(nn.Module):
 def __init__(self):
  super().__init__();self.frame_encoder=FrameCNN()
  self.temporal_encoder=nn.GRU(128,256,2,batch_first=True,bidirectional=True,dropout=.20)
  self.temporal_attention=Attn()
 def forward(self,x,return_attention=False):
  B,T,C,H,W=x.shape;f=self.frame_encoder(x.reshape(B*T,C,H,W)).reshape(B,T,-1)
  h,_=self.temporal_encoder(f);p,w=self.temporal_attention(h);return (p,w) if return_attention else p
class Model(nn.Module):
 def __init__(self):
  super().__init__();self.encoder=Encoder()
  self.classifier=nn.Sequential(nn.LayerNorm(512),nn.Dropout(.2),nn.Linear(512,256),nn.ReLU(True),nn.Dropout(.2),nn.Linear(256,44))
 def forward(self,x,return_attention=False):
  e,a=self.encoder(x,True);o=self.classifier(e);return (o,a) if return_attention else o
model=Model().to(DEVICE);model.load_state_dict(ckpt["model_state_dict"],strict=True);model.eval()
with torch.inference_mode():
 z=model(torch.zeros(1,16,3,112,112,device=DEVICE))
print("Strict load OK | logits:",tuple(z.shape));assert z.shape==(1,44)

In [ ]:
# CELL 4 — Validation data and temporal perturbations
df=pd.read_csv(MANIFEST);df["model_class_id"]=df["model_class_id"].astype(int)
val=df[df.split=="val"].reset_index(drop=True);assert len(val)==354
MEAN=np.array([.485,.456,.406],np.float32);STD=np.array([.229,.224,.225],np.float32)
def norm(x):
 x=x.astype(np.float32)/255.;x=(x-MEAN[None,None,None,:])/STD[None,None,None,:]
 return np.ascontiguousarray(np.transpose(x,(0,3,1,2)),dtype=np.float32)
class DS(Dataset):
 def __init__(self,mode):self.mode=mode
 def __len__(self):return len(val)
 def __getitem__(self,i):
  r=val.iloc[i]
  with np.load(Path(str(r.mouth_cache_path))) as q:x=q["frames"].copy()
  assert x.shape==(16,112,112,3)
  if self.mode=="reverse":x=x[::-1].copy()
  elif self.mode=="shuffle":
   rng=np.random.default_rng(SEED+i);x=x[rng.permutation(16)].copy()
  elif self.mode=="static":x=np.repeat(x[8:9],16,0)
  elif self.mode!="original":raise ValueError(self.mode)
  return torch.from_numpy(norm(x)),int(r.model_class_id)
print("Validation:",len(val))

In [ ]:
# CELL 5 — Metrics / inference
def metrics(y,p):
 pred=p.argmax(1);top=np.argsort(-p,1)[:,:3];fs=[]
 for c in range(44):
  tp=np.sum((pred==c)&(y==c));fp=np.sum((pred==c)&(y!=c));fn=np.sum((pred!=c)&(y==c))
  pr=tp/(tp+fp) if tp+fp else 0;rc=tp/(tp+fn) if tp+fn else 0
  fs.append(2*pr*rc/(pr+rc) if pr+rc else 0)
 return {"accuracy":float(np.mean(pred==y)),"macro_f1":float(np.mean(fs)),
         "top3":float(np.mean([y[i] in top[i] for i in range(len(y))])),
         "confidence":float(p.max(1).mean())}
def js(p,q,eps=1e-8):
 p=np.clip(p,eps,1);q=np.clip(q,eps,1);m=.5*(p+q)
 return .5*(np.sum(p*np.log(p/m),1)+np.sum(q*np.log(q/m),1))
@torch.inference_mode()
def infer(mode):
 loader=DataLoader(DS(mode),BATCH,shuffle=False,pin_memory=torch.cuda.is_available())
 Y=[];P=[];A=[]
 for x,y in tqdm(loader,desc=mode):
  x=x.to(DEVICE)
  with torch.amp.autocast(DEVICE.type,enabled=AMP):o,a=model(x,True)
  Y.extend(y.numpy());P.append(torch.softmax(o.float(),1).cpu().numpy());A.append(a.float().cpu().numpy())
 Y=np.asarray(Y);P=np.concatenate(P);A=np.concatenate(A)
 return Y,P,A,metrics(Y,P)

In [ ]:
# CELL 6 — Run diagnostic
modes=["original","reverse","shuffle","static"];R={}
for m in modes:
 Y,P,A,M=infer(m);R[m]={"y":Y,"p":P,"a":A,"m":M};print(m,M)

In [ ]:
# CELL 7 — Final comparison
O=R["original"];rows=[]
for m in modes:
 Q=R[m];met=Q["m"];row={"condition":m,**met}
 if m=="original":
  row.update({"top1_agreement":1.0,"prob_L1":0.0,"JS":0.0,"acc_delta":0.0,"f1_delta":0.0,"top3_delta":0.0})
 else:
  row.update({"top1_agreement":float(np.mean(O["p"].argmax(1)==Q["p"].argmax(1))),
   "prob_L1":float(np.mean(np.sum(np.abs(O["p"]-Q["p"]),1))),"JS":float(np.mean(js(O["p"],Q["p"]))),
   "acc_delta":met["accuracy"]-O["m"]["accuracy"],"f1_delta":met["macro_f1"]-O["m"]["macro_f1"],
   "top3_delta":met["top3"]-O["m"]["top3"]})
 rows.append(row)
summary=pd.DataFrame(rows);display(summary)
summary.to_csv(OUT/"04_temporal_sensitivity_summary.csv",index=False,encoding="utf-8-sig")

In [ ]:
# CELL 8 — Attention + final record
maxH=np.log(16);ar=[]
for m in modes:
 a=np.clip(R[m]["a"],1e-8,1);h=-np.sum(a*np.log(a),1)
 ar.append({"condition":m,"attention_entropy":float(h.mean()),
            "normalized_entropy":float(h.mean()/maxH),"max_attention_weight":float(a.max(1).mean())})
att=pd.DataFrame(ar);display(att);att.to_csv(OUT/"04_attention_summary.csv",index=False,encoding="utf-8-sig")
record={"split":"validation","samples":354,"conditions":summary.to_dict("records"),"attention":att.to_dict("records")}
(OUT/"04_temporal_diagnostic_final.json").write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding="utf-8")
print("="*80);print("TEMPORAL DIAGNOSTIC COMPLETE");print(summary.to_string(index=False))
print("""
อ่านผล:
- Reverse/Shuffle/Static agreement สูง + JS ต่ำ + Accuracy/F1 แทบไม่ลด
  => หลักฐานว่าโมเดลพึ่ง temporal order/motion น้อย
- Shuffle/Static ทำคะแนนตกชัด + agreement ต่ำ + JS สูง
  => temporal information มีผลจริง
Reverse อย่างเดียวไม่พอสำหรับสรุป เพราะเป็น BiGRU
""")
print("Saved:",OUT)